In [1]:
import pandas as pd
import scipy.stats as stats
from scipy.stats import binomtest
import numpy as np
import ast

In [2]:
def compute_accuracy_local(df):
    list_Trustworthiness = df["Trustworthiness"].tolist()
    list_Continuity = df["Continuity"].tolist()

    results = []
    for i in range(len(list_Trustworthiness)):
        results.append(round(0.5*list_Trustworthiness[i] + 0.5*list_Continuity[i], 7))
    return results

In [3]:
def compute_accuracy_global(df):
    list_Shephard = df["Shephard Diagram Correlation"].tolist()

    results = []
    for i in range(len(list_Shephard)):
        results.append(round(0.5*(list_Shephard[i] + 1), 7))
    return results

In [4]:
def compute_perception(df):
    list_NeighborhoodHit = df["7-Neighborhood Hit"].tolist()
    list_DistanceConsistency = df["Distance consistency"].tolist()

    results = []
    for i in range(len(list_NeighborhoodHit)):
        results.append(round(0.5*list_NeighborhoodHit[i] + 0.5*list_DistanceConsistency[i], 7))
    return results

In [5]:
def extract_K(df, K):
    # Define excluded embeddings
    excluded_embeddings = ['bert', 'bow', 'tfidf']

    # Keep only TMs (LDA, LSI, NMF) and make a copy to avoid SettingWithCopyWarning
    df_TMs = df[~df['TM'].isin(excluded_embeddings)].copy()

    # Extract the number of topics from the experiment name
    df_TMs['n_topics'] = df_TMs['Experiment'].str.extract(r'n_topics_(\d+)_')[0]

    # Keep only rows with the desired number of topics
    df_TMs = df_TMs[df_TMs['n_topics'] == str(K)]

    # Keep rows with no TMs and make a copy
    df_noTMs = df[df['TM'].isin(excluded_embeddings)].copy()
    df_noTMs['n_topics'] = str(K)

    # Combine the two DataFrames
    df_result = pd.concat([df_TMs, df_noTMs], ignore_index=True)

    return df_result

In [6]:
# Example function
def extract_hyperparam(row):
    dr_method = row['DR']
    hyperparam_str = row['Complete List of Hyperparameters']
    try:
        return hyperparam_str.split(f"'{dr_method}': ")[1]
    except IndexError:
        return None  # or "" or np.nan depending on what you prefer

In [7]:
def process_df(file, corpus, K):
    df = pd.read_csv(file)
    df["corpus"] = corpus
    df["accuracy_local"] = compute_accuracy_local(df)
    df["accuracy_global"] = compute_accuracy_global(df)
    df["perception"] = compute_perception(df)
    df_result = extract_K(df, K)
    df_selected = df_result[['Experiment','Complete List of Hyperparameters', 'DR', 'TM', 'corpus', 'accuracy_local', 'accuracy_global', 'perception', 'n_topics']]
    # Apply row-wise
    df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)
    return df_selected

In [8]:
df_20Newsgroups = process_df("results_cluster/cur_res/full_res_20_newsgroups.csv", "20Newsgroups", K = 20)
print(set(df_20Newsgroups["DR"].tolist()))
print(len(set(df_20Newsgroups["TM"].tolist())))
print(df_20Newsgroups.shape)
df_20Newsgroups.head(8)

{'som', 'tsne', 'umap'}
13
(3767, 10)


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_7216\1611633390.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)


,Experiment,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics,Hyperparameters_DR
0,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.945445,0.637502,0.175611,20,"{'n': 25, 'm': 5}}"
1,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.966916,0.733167,0.256124,20,"{'n': 15, 'm': 20}}"
2,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.968795,0.722859,0.271983,20,"{'n': 20, 'm': 25}}"
3,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.965036,0.710497,0.254861,20,"{'n': 20, 'm': 15}}"
4,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.962456,0.759466,0.237828,20,"{'n': 10, 'm': 15}}"
5,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.963677,0.696409,0.249545,20,"{'n': 25, 'm': 10}}"
6,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.963681,0.745391,0.239564,20,"{'n': 15, 'm': 15}}"
7,20_newsgroups_lda_linear_combined_n_topics_20_...,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.966959,0.734412,0.249154,20,"{'n': 10, 'm': 25}}"


In [9]:
df_Emails = process_df("results_cluster/cur_res/full_res_Emails.csv", "Emails", K = 8)
print(set(df_Emails["DR"].tolist()))
print(len(set(df_Emails["TM"].tolist())))
print(df_Emails.shape)
df_Emails.head(8)

{'mds', 'som', 'tsne', 'umap'}
12
(3622, 10)


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_7216\1611633390.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)


,Experiment,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics,Hyperparameters_DR
0,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Emails,0.549153,0.349980,0.365931,8,"{'max_iter': 300, 'dissimilarity_metric': 'jen..."
1,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Emails,0.549153,0.349980,0.365931,8,"{'max_iter': 150, 'dissimilarity_metric': 'jen..."
2,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Emails,0.549153,0.349980,0.365931,8,"{'max_iter': 100, 'dissimilarity_metric': 'jen..."
3,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Emails,0.549153,0.349980,0.365931,8,"{'max_iter': 200, 'dissimilarity_metric': 'jen..."
4,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Emails,0.549153,0.349980,0.365931,8,"{'max_iter': 250, 'dissimilarity_metric': 'jen..."
5,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",som,lda_linear_combined,Emails,0.975945,0.742593,0.358977,8,"{'n': 15, 'm': 10}}"
6,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",som,lda_linear_combined,Emails,0.981486,0.754598,0.387506,8,"{'n': 15, 'm': 15}}"
7,emails_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",som,lda_linear_combined,Emails,0.978387,0.715615,0.371372,8,"{'n': 25, 'm': 10}}"


In [10]:
df_BBC = process_df("results_cluster/cur_res/full_res_bbc_news.csv", "BBC", K = 10)
print(set(df_BBC["DR"].tolist()))
print(len(set(df_BBC["TM"].tolist())))
print(df_BBC.shape)
df_BBC.head(8)

{'mds', 'som', 'tsne', 'umap'}
13
(3834, 10)


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_7216\1611633390.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)


,Experiment,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics,Hyperparameters_DR
0,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,BBC,0.555295,0.348777,0.279711,10,"{'max_iter': 150, 'dissimilarity_metric': 'jen..."
1,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,BBC,0.555295,0.348777,0.279711,10,"{'max_iter': 250, 'dissimilarity_metric': 'jen..."
2,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,BBC,0.555295,0.348777,0.279711,10,"{'max_iter': 300, 'dissimilarity_metric': 'jen..."
3,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,BBC,0.555295,0.348777,0.279711,10,"{'max_iter': 100, 'dissimilarity_metric': 'jen..."
4,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,BBC,0.977098,0.732741,0.461027,10,"{'n': 25, 'm': 25}}"
5,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,BBC,0.971091,0.752752,0.393676,10,"{'n': 10, 'm': 20}}"
6,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,BBC,0.972679,0.793627,0.451043,10,"{'n': 10, 'm': 30}}"
7,bbc_news_lda_linear_combined_n_topics_10_alpha...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,BBC,0.949498,0.716631,0.349631,10,"{'n': 15, 'm': 5}}"


In [11]:
df_lyrics = process_df("results_cluster/cur_res/full_res_lyrics.csv", "Lyrics", K = 8)
print(set(df_lyrics["DR"].tolist()))
print(len(set(df_lyrics["TM"].tolist())))
print(df_lyrics.shape)
df_lyrics.head(8)

{'mds', 'som', 'tsne', 'umap'}
13
(3849, 10)


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_7216\1611633390.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)


,Experiment,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics,Hyperparameters_DR
0,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Lyrics,0.553510,0.383539,0.309082,8,"{'max_iter': 150, 'dissimilarity_metric': 'jen..."
1,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Lyrics,0.553510,0.383539,0.309082,8,"{'max_iter': 200, 'dissimilarity_metric': 'jen..."
2,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Lyrics,0.553510,0.383539,0.309082,8,"{'max_iter': 100, 'dissimilarity_metric': 'jen..."
3,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Lyrics,0.553510,0.383539,0.309082,8,"{'max_iter': 300, 'dissimilarity_metric': 'jen..."
4,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",mds,lda_linear_combined,Lyrics,0.553510,0.383539,0.309082,8,"{'max_iter': 250, 'dissimilarity_metric': 'jen..."
5,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",som,lda_linear_combined,Lyrics,0.980180,0.865047,0.418788,8,"{'n': 25, 'm': 10}}"
6,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",som,lda_linear_combined,Lyrics,0.983409,0.849465,0.458111,8,"{'n': 25, 'm': 20}}"
7,lyrics_lda_linear_combined_n_topics_8_alpha_au...,"{'lda': {'n_topics': 8, 'alpha': 'auto', 'iter...",som,lda_linear_combined,Lyrics,0.660572,0.661142,0.403547,8,"{'n': 15, 'm': 15}}"


In [12]:
df_Reuters = process_df("results_cluster/cur_res/full_res_reuters.csv", "Reuters", K = 10)
print(set(df_Reuters["DR"].tolist()))
print(len(set(df_Reuters["TM"].tolist())))
print(df_Reuters.shape)
df_Reuters.head(8)

{'mds', 'som', 'tsne', 'umap'}
13
(3849, 10)


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_7216\1611633390.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)


,Experiment,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics,Hyperparameters_DR
0,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,Reuters,0.534701,0.433668,0.224489,10,"{'max_iter': 300, 'dissimilarity_metric': 'jen..."
1,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,Reuters,0.534701,0.433668,0.221359,10,"{'max_iter': 200, 'dissimilarity_metric': 'jen..."
2,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,Reuters,0.534701,0.433668,0.223907,10,"{'max_iter': 100, 'dissimilarity_metric': 'jen..."
3,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,Reuters,0.534701,0.433668,0.227430,10,"{'max_iter': 250, 'dissimilarity_metric': 'jen..."
4,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",mds,lda_linear_combined,Reuters,0.534701,0.433668,0.228164,10,"{'max_iter': 150, 'dissimilarity_metric': 'jen..."
5,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,Reuters,0.952695,0.708723,0.183811,10,"{'n': 20, 'm': 5}}"
6,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,Reuters,0.972765,0.828730,0.199835,10,"{'n': 15, 'm': 30}}"
7,reuters_lda_linear_combined_n_topics_10_alpha_...,"{'lda': {'n_topics': 10, 'alpha': 'auto', 'ite...",som,lda_linear_combined,Reuters,0.973764,0.863262,0.196026,10,"{'n': 25, 'm': 20}}"


In [13]:
df_7Categories = process_df("results_cluster/cur_res/full_res_seven_categories.csv", "7Categories", K = 14)
print(set(df_7Categories["DR"].tolist()))
print(len(set(df_7Categories["TM"].tolist())))
print(df_7Categories.shape)
df_7Categories.head(8)

{'som', 'tsne', 'umap'}
13
(3754, 10)


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_7216\1611633390.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Hyperparameters_DR'] = df_selected.apply(extract_hyperparam, axis=1)


,Experiment,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics,Hyperparameters_DR
0,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.975927,0.747188,0.510850,14,"{'n': 30, 'm': 10}}"
1,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.981486,0.861864,0.512723,14,"{'n': 20, 'm': 25}}"
2,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.981540,0.810903,0.548495,14,"{'n': 30, 'm': 25}}"
3,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.976371,0.790712,0.505322,14,"{'n': 25, 'm': 10}}"
4,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.977848,0.714657,0.497305,14,"{'n': 30, 'm': 15}}"
5,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.968817,0.784638,0.422975,14,"{'n': 5, 'm': 20}}"
6,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.977952,0.861598,0.500982,14,"{'n': 15, 'm': 15}}"
7,seven_categories_lda_linear_combined_n_topics_...,"{'lda': {'n_topics': 14, 'alpha': 'auto', 'ite...",som,lda_linear_combined,7Categories,0.974511,0.816426,0.487962,14,"{'n': 20, 'm': 10}}"


In [14]:
df_all_corpora = pd.concat([df_20Newsgroups, df_Emails, df_BBC, df_lyrics, df_Reuters, df_7Categories], ignore_index=True)
df_all_corpora.shape

(22675, 10)

## Binary Test 1: Tfidf Weighting Scheme

In [15]:
df = df_all_corpora

In [16]:
# concerning local accuracy alpha
# Define term model pairs to compare

tm_pairs = {
    'bow': 'tfidf',
    'lsi': 'lsi_tfidf',
    'lsi_linear_combined': 'lsi_tfidf_linear_combined',
    'nmf': 'nmf_tfidf',
    'nmf_linear_combined': 'nmf_tfidf_linear_combined'
}

group_keys = ['corpus', 'Hyperparameters_DR']  # Group by corpus and hyperparams only since DR is looped separately

results = []

for tm1, tm2 in tm_pairs.items():
    print(f"\n=== Comparing '{tm1}' vs. '{tm2}' ===")

    # Filter by TM
    df_tm1_all = df[df['TM'] == tm1].copy()
    df_tm2_all = df[df['TM'] == tm2].copy()

    # Find DR values present in both datasets
    common_drs = set(df_tm1_all['DR']).intersection(df_tm2_all['DR'])
    print(f"Common DR methods: {sorted(common_drs)}")

    for dr in sorted(common_drs):
        print(f"\n--- TM pair '{tm1}' vs '{tm2}' for DR='{dr}' ---")

        # Filter data for this DR
        df_tm1 = df_tm1_all[df_tm1_all['DR'] == dr]
        df_tm2 = df_tm2_all[df_tm2_all['DR'] == dr]

        grouped_tm1 = df_tm1.groupby(group_keys)
        grouped_tm2 = df_tm2.groupby(group_keys)

        common_groups = set(grouped_tm1.groups.keys()) & set(grouped_tm2.groups.keys())
        print(f"Matched groups (by corpus & Hyperparameters_DR): {len(common_groups)}")

        n = 0
        n_improved = 0

        for group in common_groups:
            rows_tm1 = grouped_tm1.get_group(group)
            rows_tm2 = grouped_tm2.get_group(group)

            if len(rows_tm1) != 1 or len(rows_tm2) != 1:
                print(f"Skipping group {group} due to multiple entries")
                continue

            row_tm1 = rows_tm1.iloc[0]
            row_tm2 = rows_tm2.iloc[0]

            exp1 = row_tm1['Experiment']
            exp2 = row_tm2['Experiment']
            acc1 = row_tm1['accuracy_local']
            acc2 = row_tm2['accuracy_local']
            hyper1 = row_tm1.get('Complete List of Hyperparameters', 'N/A')
            hyper2 = row_tm2.get('Complete List of Hyperparameters', 'N/A')

            improved = acc2 > acc1
            if improved:
                n_improved += 1
            n += 1

            status = "Improved" if improved else "Not improved"
            #print(f"\nComparing group {group}:")
            #print(f" - {tm1} Experiment: {exp1}")
            #print(f"   accuracy_local: {acc1:.4f}")
            #print(f"   Complete List of Hyperparameters: {hyper1}")
            #print(f" - {tm2} Experiment: {exp2}")
            #print(f"   accuracy_local: {acc2:.4f}")
            #print(f"   Complete List of Hyperparameters: {hyper2}")
            #print(f" -> {status}")

        if n > 0:
            p_value = binomtest(n_improved, n, p=0.5, alternative='greater').pvalue
        else:
            p_value = None

        print(f"\nSummary for TM pair '{tm1}' vs '{tm2}', DR='{dr}':")
        print(f"  Total comparisons: {n}")
        print(f"  Improvements: {n_improved}")
        if p_value is not None:
            print(f"  One-sided binomial test p-value: {p_value:.4g}")
        else:
            print("  No valid comparisons to test.")

        results.append({
            'TM Pair': f"{tm1} vs {tm2}",
            'DR': dr,
            'n': n,
            'n_improved': n_improved,
            'p_value': p_value
        })

print("\n=== Final Summary ===")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))



=== Comparing 'bow' vs. 'tfidf' ===
Common DR methods: ['mds', 'som', 'tsne', 'umap']

--- TM pair 'bow' vs 'tfidf' for DR='mds' ---
Matched groups (by corpus & Hyperparameters_DR): 15

Summary for TM pair 'bow' vs 'tfidf', DR='mds':
  Total comparisons: 15
  Improvements: 10
  One-sided binomial test p-value: 0.1509

--- TM pair 'bow' vs 'tfidf' for DR='som' ---
Matched groups (by corpus & Hyperparameters_DR): 180

Summary for TM pair 'bow' vs 'tfidf', DR='som':
  Total comparisons: 180
  Improvements: 177
  One-sided binomial test p-value: 6.344e-49

--- TM pair 'bow' vs 'tfidf' for DR='tsne' ---
Matched groups (by corpus & Hyperparameters_DR): 720

Summary for TM pair 'bow' vs 'tfidf', DR='tsne':
  Total comparisons: 720
  Improvements: 313
  One-sided binomial test p-value: 0.9998

--- TM pair 'bow' vs 'tfidf' for DR='umap' ---
Matched groups (by corpus & Hyperparameters_DR): 210

Summary for TM pair 'bow' vs 'tfidf', DR='umap':
  Total comparisons: 210
  Improvements: 188
  One-s

In [17]:
def compare_tm_pairs(df, value_col):
    tm_pairs = {
        'bow': 'tfidf',
        'lsi': 'lsi_tfidf',
        'lsi_linear_combined': 'lsi_tfidf_linear_combined',
        'nmf': 'nmf_tfidf',
        'nmf_linear_combined': 'nmf_tfidf_linear_combined'
    }

    group_keys = ['corpus', 'Hyperparameters_DR']
    results = []

    for tm1, tm2 in tm_pairs.items():
        df_tm1_all = df[df['TM'] == tm1].copy()
        df_tm2_all = df[df['TM'] == tm2].copy()
        common_drs = set(df_tm1_all['DR']).intersection(df_tm2_all['DR'])

        for dr in sorted(common_drs):
            df_tm1 = df_tm1_all[df_tm1_all['DR'] == dr]
            df_tm2 = df_tm2_all[df_tm2_all['DR'] == dr]

            grouped_tm1 = df_tm1.groupby(group_keys)
            grouped_tm2 = df_tm2.groupby(group_keys)
            common_groups = set(grouped_tm1.groups.keys()) & set(grouped_tm2.groups.keys())

            n = 0
            n_improved = 0

            for group in common_groups:
                rows_tm1 = grouped_tm1.get_group(group)
                rows_tm2 = grouped_tm2.get_group(group)

                if len(rows_tm1) != 1 or len(rows_tm2) != 1:
                    continue

                val1 = rows_tm1.iloc[0][value_col]
                val2 = rows_tm2.iloc[0][value_col]

                if val2 > val1:
                    n_improved += 1
                n += 1

            p_value = round(binomtest(n_improved, n, p=0.5, alternative='greater').pvalue if n > 0 else None, 2)

            results.append({
                'TM Pair': f"{tm1} vs {tm2}",
                'DR': dr,
                'n': n,
                'n_improved': n_improved,
                'p_value': p_value
            })

    results_df = pd.DataFrame(results)
    print("\n=== Final Summary ===")
    print(results_df.to_string(index=False))
    return results_df


In [18]:
compare_tm_pairs(df = df_all_corpora, value_col = "accuracy_local")


=== Final Summary ===
                                         TM Pair   DR   n  n_improved  p_value
                                    bow vs tfidf  mds  15          10     0.15
                                    bow vs tfidf  som 180         177     0.00
                                    bow vs tfidf tsne 720         313     1.00
                                    bow vs tfidf umap 210         188     0.00
                                lsi vs lsi_tfidf  mds  19          10     0.50
                                lsi vs lsi_tfidf  som 216         201     0.00
                                lsi vs lsi_tfidf tsne 849         753     0.00
                                lsi vs lsi_tfidf umap 247         235     0.00
lsi_linear_combined vs lsi_tfidf_linear_combined  mds  19          10     0.50
lsi_linear_combined vs lsi_tfidf_linear_combined  som 215         199     0.00
lsi_linear_combined vs lsi_tfidf_linear_combined tsne 854         760     0.00
lsi_linear_combined vs lsi_tf

,TM Pair,DR,n,n_improved,p_value
0,bow vs tfidf,mds,15,10,0.15
1,bow vs tfidf,som,180,177,0.00
2,bow vs tfidf,tsne,720,313,1.00
3,bow vs tfidf,umap,210,188,0.00
4,lsi vs lsi_tfidf,mds,19,10,0.50
5,lsi vs lsi_tfidf,som,216,201,0.00
6,lsi vs lsi_tfidf,tsne,849,753,0.00
7,lsi vs lsi_tfidf,umap,247,235,0.00
8,lsi_linear_combined vs lsi_tfidf_linear_combined,mds,19,10,0.50
9,lsi_linear_combined vs lsi_tfidf_linear_combined,som,215,199,0.00


In [19]:
compare_tm_pairs(df = df_all_corpora, value_col = "accuracy_global")


=== Final Summary ===
                                         TM Pair   DR   n  n_improved  p_value
                                    bow vs tfidf  mds  15          15     0.00
                                    bow vs tfidf  som 180         124     0.00
                                    bow vs tfidf tsne 720          45     1.00
                                    bow vs tfidf umap 210         132     0.00
                                lsi vs lsi_tfidf  mds  19           5     0.99
                                lsi vs lsi_tfidf  som 216         135     0.00
                                lsi vs lsi_tfidf tsne 849         368     1.00
                                lsi vs lsi_tfidf umap 247         229     0.00
lsi_linear_combined vs lsi_tfidf_linear_combined  mds  19           5     0.99
lsi_linear_combined vs lsi_tfidf_linear_combined  som 215         138     0.00
lsi_linear_combined vs lsi_tfidf_linear_combined tsne 854         366     1.00
lsi_linear_combined vs lsi_tf

,TM Pair,DR,n,n_improved,p_value
0,bow vs tfidf,mds,15,15,0.00
1,bow vs tfidf,som,180,124,0.00
2,bow vs tfidf,tsne,720,45,1.00
3,bow vs tfidf,umap,210,132,0.00
4,lsi vs lsi_tfidf,mds,19,5,0.99
5,lsi vs lsi_tfidf,som,216,135,0.00
6,lsi vs lsi_tfidf,tsne,849,368,1.00
7,lsi vs lsi_tfidf,umap,247,229,0.00
8,lsi_linear_combined vs lsi_tfidf_linear_combined,mds,19,5,0.99
9,lsi_linear_combined vs lsi_tfidf_linear_combined,som,215,138,0.00


In [20]:
compare_tm_pairs(df = df_all_corpora, value_col = "perception")


=== Final Summary ===
                                         TM Pair   DR   n  n_improved  p_value
                                    bow vs tfidf  mds  15           2     1.00
                                    bow vs tfidf  som 180         142     0.00
                                    bow vs tfidf tsne 720         444     0.00
                                    bow vs tfidf umap 210         154     0.00
                                lsi vs lsi_tfidf  mds  19           8     0.82
                                lsi vs lsi_tfidf  som 216         184     0.00
                                lsi vs lsi_tfidf tsne 849         668     0.00
                                lsi vs lsi_tfidf umap 247         216     0.00
lsi_linear_combined vs lsi_tfidf_linear_combined  mds  19           8     0.82
lsi_linear_combined vs lsi_tfidf_linear_combined  som 215         169     0.00
lsi_linear_combined vs lsi_tfidf_linear_combined tsne 854         661     0.00
lsi_linear_combined vs lsi_tf

,TM Pair,DR,n,n_improved,p_value
0,bow vs tfidf,mds,15,2,1.00
1,bow vs tfidf,som,180,142,0.00
2,bow vs tfidf,tsne,720,444,0.00
3,bow vs tfidf,umap,210,154,0.00
4,lsi vs lsi_tfidf,mds,19,8,0.82
5,lsi vs lsi_tfidf,som,216,184,0.00
6,lsi vs lsi_tfidf,tsne,849,668,0.00
7,lsi vs lsi_tfidf,umap,247,216,0.00
8,lsi_linear_combined vs lsi_tfidf_linear_combined,mds,19,8,0.82
9,lsi_linear_combined vs lsi_tfidf_linear_combined,som,215,169,0.00


## Binary Test 2: Star Coordinates

In [21]:
def compare_tm_pairs_SC(df, value_col):
    tm_pairs = {
        'lsi': 'lsi_linear_combined',
        'lsi_tfidf': 'lsi_tfidf_linear_combined',
        'nmf': 'nmf_linear_combined',
        'nmf_tfidf': 'nmf_tfidf_linear_combined',
        'lda': 'lda_linear_combined'
    }

    group_keys = ['corpus', 'Hyperparameters_DR']
    results = []

    for tm1, tm2 in tm_pairs.items():
        df_tm1_all = df[df['TM'] == tm1].copy()
        df_tm2_all = df[df['TM'] == tm2].copy()
        common_drs = set(df_tm1_all['DR']).intersection(df_tm2_all['DR'])

        for dr in sorted(common_drs):
            df_tm1 = df_tm1_all[df_tm1_all['DR'] == dr]
            df_tm2 = df_tm2_all[df_tm2_all['DR'] == dr]

            grouped_tm1 = df_tm1.groupby(group_keys)
            grouped_tm2 = df_tm2.groupby(group_keys)
            common_groups = set(grouped_tm1.groups.keys()) & set(grouped_tm2.groups.keys())

            n = 0
            n_improved = 0

            for group in common_groups:
                rows_tm1 = grouped_tm1.get_group(group)
                rows_tm2 = grouped_tm2.get_group(group)

                if len(rows_tm1) != 1 or len(rows_tm2) != 1:
                    continue

                val1 = rows_tm1.iloc[0][value_col]
                val2 = rows_tm2.iloc[0][value_col]

                if val2 > val1:
                    n_improved += 1
                n += 1

            p_value = round(binomtest(n_improved, n, p=0.5, alternative='greater').pvalue if n > 0 else None, 2)

            results.append({
                'TM Pair': f"{tm1} vs {tm2}",
                'DR': dr,
                'n': n,
                'n_improved': n_improved,
                'p_value': p_value
            })

    results_df = pd.DataFrame(results)
    print("\n=== Final Summary ===")
    print(results_df.to_string(index=False))
    return results_df


In [22]:
compare_tm_pairs_SC(df = df_all_corpora, value_col = 'accuracy_local')


=== Final Summary ===
                               TM Pair   DR   n  n_improved  p_value
            lsi vs lsi_linear_combined  mds  19          14     0.03
            lsi vs lsi_linear_combined  som 216         133     0.00
            lsi vs lsi_linear_combined tsne 853         309     1.00
            lsi vs lsi_linear_combined umap 251         139     0.05
lsi_tfidf vs lsi_tfidf_linear_combined  mds  19           0     1.00
lsi_tfidf vs lsi_tfidf_linear_combined  som 215         159     0.00
lsi_tfidf vs lsi_tfidf_linear_combined tsne 850         273     1.00
lsi_tfidf vs lsi_tfidf_linear_combined umap 248         101     1.00
            nmf vs nmf_linear_combined  mds  19           9     0.68
            nmf vs nmf_linear_combined  som 216          89     1.00
            nmf vs nmf_linear_combined tsne 855         745     0.00
            nmf vs nmf_linear_combined umap 246         230     0.00
nmf_tfidf vs nmf_tfidf_linear_combined  mds  19           9     0.68
nmf_tfidf v

,TM Pair,DR,n,n_improved,p_value
0,lsi vs lsi_linear_combined,mds,19,14,0.03
1,lsi vs lsi_linear_combined,som,216,133,0.00
2,lsi vs lsi_linear_combined,tsne,853,309,1.00
3,lsi vs lsi_linear_combined,umap,251,139,0.05
4,lsi_tfidf vs lsi_tfidf_linear_combined,mds,19,0,1.00
5,lsi_tfidf vs lsi_tfidf_linear_combined,som,215,159,0.00
6,lsi_tfidf vs lsi_tfidf_linear_combined,tsne,850,273,1.00
7,lsi_tfidf vs lsi_tfidf_linear_combined,umap,248,101,1.00
8,nmf vs nmf_linear_combined,mds,19,9,0.68
9,nmf vs nmf_linear_combined,som,216,89,1.00


In [23]:
compare_tm_pairs_SC(df = df_all_corpora, value_col = 'accuracy_global')


=== Final Summary ===
                               TM Pair   DR   n  n_improved  p_value
            lsi vs lsi_linear_combined  mds  19           0     1.00
            lsi vs lsi_linear_combined  som 216         122     0.03
            lsi vs lsi_linear_combined tsne 853         168     1.00
            lsi vs lsi_linear_combined umap 251         126     0.50
lsi_tfidf vs lsi_tfidf_linear_combined  mds  19           0     1.00
lsi_tfidf vs lsi_tfidf_linear_combined  som 215         118     0.09
lsi_tfidf vs lsi_tfidf_linear_combined tsne 850         173     1.00
lsi_tfidf vs lsi_tfidf_linear_combined umap 248         124     0.53
            nmf vs nmf_linear_combined  mds  19          15     0.01
            nmf vs nmf_linear_combined  som 216         166     0.00
            nmf vs nmf_linear_combined tsne 855         798     0.00
            nmf vs nmf_linear_combined umap 246         239     0.00
nmf_tfidf vs nmf_tfidf_linear_combined  mds  19          10     0.50
nmf_tfidf v

,TM Pair,DR,n,n_improved,p_value
0,lsi vs lsi_linear_combined,mds,19,0,1.00
1,lsi vs lsi_linear_combined,som,216,122,0.03
2,lsi vs lsi_linear_combined,tsne,853,168,1.00
3,lsi vs lsi_linear_combined,umap,251,126,0.50
4,lsi_tfidf vs lsi_tfidf_linear_combined,mds,19,0,1.00
5,lsi_tfidf vs lsi_tfidf_linear_combined,som,215,118,0.09
6,lsi_tfidf vs lsi_tfidf_linear_combined,tsne,850,173,1.00
7,lsi_tfidf vs lsi_tfidf_linear_combined,umap,248,124,0.53
8,nmf vs nmf_linear_combined,mds,19,15,0.01
9,nmf vs nmf_linear_combined,som,216,166,0.00


In [24]:
compare_tm_pairs_SC(df = df_all_corpora, value_col = 'perception')


=== Final Summary ===
                               TM Pair   DR   n  n_improved  p_value
            lsi vs lsi_linear_combined  mds  19           2     1.00
            lsi vs lsi_linear_combined  som 216         110     0.42
            lsi vs lsi_linear_combined tsne 853         177     1.00
            lsi vs lsi_linear_combined umap 251         119     0.81
lsi_tfidf vs lsi_tfidf_linear_combined  mds  19           2     1.00
lsi_tfidf vs lsi_tfidf_linear_combined  som 215          98     0.91
lsi_tfidf vs lsi_tfidf_linear_combined tsne 850         186     1.00
lsi_tfidf vs lsi_tfidf_linear_combined umap 248         123     0.58
            nmf vs nmf_linear_combined  mds  19          10     0.50
            nmf vs nmf_linear_combined  som 216         124     0.02
            nmf vs nmf_linear_combined tsne 855         467     0.00
            nmf vs nmf_linear_combined umap 246          72     1.00
nmf_tfidf vs nmf_tfidf_linear_combined  mds  19          12     0.18
nmf_tfidf v

,TM Pair,DR,n,n_improved,p_value
0,lsi vs lsi_linear_combined,mds,19,2,1.00
1,lsi vs lsi_linear_combined,som,216,110,0.42
2,lsi vs lsi_linear_combined,tsne,853,177,1.00
3,lsi vs lsi_linear_combined,umap,251,119,0.81
4,lsi_tfidf vs lsi_tfidf_linear_combined,mds,19,2,1.00
5,lsi_tfidf vs lsi_tfidf_linear_combined,som,215,98,0.91
6,lsi_tfidf vs lsi_tfidf_linear_combined,tsne,850,186,1.00
7,lsi_tfidf vs lsi_tfidf_linear_combined,umap,248,123,0.58
8,nmf vs nmf_linear_combined,mds,19,10,0.50
9,nmf vs nmf_linear_combined,som,216,124,0.02
